# 07 — Mars candidate follow-up

Notebook 06 produced **candidates**, not confirmed processes. This notebook asks what the
available independent observations actually support.

- Dust candidates: test whether albedo and thermal inertia independently support a fine-particle surface.
- Sulfate candidates: find nearby CRISM observations and summarize small hydration browse products.
- Hydrogen candidates: inventory nearby SHARAD observations for later radar analysis.

A coverage hit is never treated as scientific confirmation. CRISM browse colors are never treated as a
positive mineral identification. A dust-compatible surface is not labelled active wind transport.
Earth matching remains closed until a real process, rather than only an environment, passes its required evidence.


In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import math
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 100)


@dataclass(frozen=True)
class ValidationConfig:
    queue_path_override: str | None = None
    regions_path_override: str | None = None
    output_root_override: str | None = None
    ode_base_url: str = "https://oderest.rsl.wustl.edu/live2"
    request_timeout_seconds: int = 45
    query_pause_seconds: float = 0.15
    crism_search_radius_degrees: float = 1.5
    radar_search_radius_degrees: float = 5.0
    maximum_crism_products_per_candidate: int = 1
    maximum_radar_ids_per_query: int = 20
    minimum_thermal_coverage: float = 0.90
    minimum_albedo_coverage: float = 0.90
    maximum_absolute_latitude_for_dust_test: float = 60.0
    dust_albedo_minimum: float = 0.25
    dust_thermal_inertia_minimum: float = 20.0
    dust_thermal_inertia_maximum: float = 100.0
    contradictory_thermal_inertia_minimum: float = 200.0
    minimum_supported_examples_for_distribution: int = 5
    minimum_spatial_bins_for_distribution: int = 3


CFG = ValidationConfig()
RUN_ROOT = Path(CFG.output_root_override) if CFG.output_root_override else (
    Path("/kaggle/working/mars_candidate_validation")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "build" / "mars_candidate_validation"
)
DATA_DIR = RUN_ROOT / "downloaded"
FIGURE_DIR = RUN_ROOT / "figures"
for directory in (RUN_ROOT, DATA_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {RUN_ROOT}")


In [ ]:
def discover_file(filename: str, override: str | None, required_parent: str) -> Path:
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"Configured path does not exist: {path}")
        return path

    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path("/tmp")]
    matches: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob(filename):
            path_text = path.as_posix()
            if required_parent in path_text and RUN_ROOT.as_posix() not in path_text:
                matches.append(path)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach the required upstream Kaggle notebook output "
            f"or set its path override. Expected a path containing '{required_parent}'."
        )
    matches = sorted(set(matches), key=lambda p: ("/kaggle/input/" not in p.as_posix(), len(p.as_posix())))
    return matches[0]


QUEUE_PATH = discover_file(
    "mars_process_review_queue.parquet", CFG.queue_path_override, "mars_process_validation"
)
REGIONS_PATH = discover_file(
    "mars_regions.parquet", CFG.regions_path_override, "mars_region_discovery"
)
review_queue_df = pd.read_parquet(QUEUE_PATH)
regions_df = pd.read_parquet(REGIONS_PATH)

required_queue_columns = {
    "process_id", "candidate_label", "region_id", "scale", "centroid_latitude",
    "centroid_longitude", "screening_score", "missing_for_validation", "status",
}
required_region_columns = {
    "region_id", "raw_mean__thermal_inertia_mean", "raw_mean__albedo_mean",
    "raw_mean__quality_masked__mineral_surface_dust",
    "quality_mean__thermal_inertia_coverage", "quality_mean__albedo_coverage",
    "quality_mean__polar_metric_limit",
}
assert required_queue_columns <= set(review_queue_df.columns), required_queue_columns - set(review_queue_df.columns)
assert required_region_columns <= set(regions_df.columns), required_region_columns - set(regions_df.columns)
assert regions_df["region_id"].is_unique, "Region IDs must be unique before candidate enrichment"
assert len(review_queue_df) > 0, "Notebook 06 produced an empty review queue"

candidates_df = review_queue_df.merge(
    regions_df[list(required_region_columns)], on="region_id", how="left", validate="many_to_one"
)
candidates_df.insert(0, "candidate_row_id", [f"candidate_{index:04d}" for index in range(len(candidates_df))])
assert candidates_df["candidate_row_id"].is_unique

print(f"Queue: {QUEUE_PATH}")
print(f"Regions: {REGIONS_PATH}")
display(candidates_df.groupby(["process_id", "candidate_label"], observed=True).size().rename("rows").reset_index())


## What counts as evidence here?

The table below is the notebook's contract. A number is only used for the claim supported by its cited source.
Instrument coverage merely tells us whether a stronger follow-up is possible.


In [ ]:
validation_rules_df = pd.DataFrame([
    {
        "rule_id": "dust_thermophysical_support",
    "candidate_label": "dust_mantling_candidate",
    "evidence_used": "TES albedo and thermal inertia (separate variables, but not an instrument-independent test)",
        "executable_rule": "albedo > 0.25 and 20 <= thermal inertia <= 100, with adequate coverage outside polar limits",
        "claim_allowed": "surface thermophysically compatible with fine particulate dust",
        "claim_not_allowed": "active aeolian transport or a formation mechanism",
        "citation": "Lange et al. (2022), doi:10.1029/2022JE007365; Ruff & Christensen (2002), doi:10.1029/2001JE001580",
        "source_url": "https://doi.org/10.1029/2022JE007365",
    },
    {
        "rule_id": "dust_thermophysical_contradiction",
    "candidate_label": "dust_mantling_candidate",
    "evidence_used": "TES albedo and thermal inertia (separate variables, but not an instrument-independent test)",
        "executable_rule": "albedo < 0.25 and thermal inertia > 200, with adequate coverage outside polar limits",
        "claim_allowed": "current simple dust interpretation is contradicted by the thermophysical combination",
        "claim_not_allowed": "dust is absent at every thickness or spatial scale",
        "citation": "Ruff & Christensen (2002), doi:10.1029/2001JE001580",
        "source_url": "https://doi.org/10.1029/2001JE001580",
    },
    {
        "rule_id": "crism_hydration_browse_triage",
        "candidate_label": "TES_sulfate_followup_candidate",
        "evidence_used": "CRISM MTRDR HYD browse parameters SINDEX2, BD2100_2 and BD1900_2",
        "executable_rule": "summarize valid parameter values and joint scene-relative upper-tail pixels",
        "claim_allowed": "prioritize a scene for full spectral inspection",
        "claim_not_allowed": "positive sulfate or hydrated-mineral identification",
        "citation": "Viviano-Beck et al. (2014), doi:10.1002/2014JE004627; CRISM MTRDR product label",
        "source_url": "https://doi.org/10.1002/2014JE004627",
    },
    {
        "rule_id": "sharad_coverage_inventory",
        "candidate_label": "hydrogen_enrichment_candidate",
        "evidence_used": "ODE inventory of nearby SHARAD USRDRV2 and SHSIMU observations",
        "executable_rule": "record intersecting observation counts and product IDs",
        "claim_allowed": "radar follow-up is available or unavailable",
        "claim_not_allowed": "subsurface ice, interface, thickness or composition",
        "citation": "Seu et al. (2007), doi:10.1029/2006JE002745; PDS SHARAD archive",
        "source_url": "https://doi.org/10.1029/2006JE002745",
    },
])
display(validation_rules_df[["rule_id", "claim_allowed", "claim_not_allowed", "citation"]])


## 1. Separate thermophysical dust check

Notebook 06 used a dust-related mineral proxy to propose these regions. Here we test different measurements
from the same broad TES product family (so this is not a fully instrument-independent confirmation):
a bright surface with low thermal inertia is compatible with fine loose material. Missing or polar-limited
measurements stay unresolved. Even a supported result describes the **surface environment**, not present-day wind activity.


In [ ]:
def validate_dust_candidates(frame: pd.DataFrame) -> pd.DataFrame:
    dust = frame.loc[frame["candidate_label"].eq("dust_mantling_candidate")].copy()
    albedo = dust["raw_mean__albedo_mean"]
    inertia = dust["raw_mean__thermal_inertia_mean"]
    adequate = (
        albedo.notna()
        & inertia.notna()
        & dust["quality_mean__albedo_coverage"].ge(CFG.minimum_albedo_coverage)
        & dust["quality_mean__thermal_inertia_coverage"].ge(CFG.minimum_thermal_coverage)
        & dust["quality_mean__polar_metric_limit"].fillna(1).lt(0.5)
        & dust["centroid_latitude"].abs().lt(CFG.maximum_absolute_latitude_for_dust_test)
    )
    supports = (
        adequate
        & albedo.gt(CFG.dust_albedo_minimum)
        & inertia.between(CFG.dust_thermal_inertia_minimum, CFG.dust_thermal_inertia_maximum, inclusive="both")
    )
    contradicts = (
        adequate
        & albedo.lt(CFG.dust_albedo_minimum)
        & inertia.gt(CFG.contradictory_thermal_inertia_minimum)
    )
    dust["validation_verdict"] = np.select(
        [supports, contradicts], ["supported", "rejected"], default="inconclusive"
    )
    dust["validated_interpretation"] = np.select(
        [supports, contradicts],
        [
            "fine_particulate_dust_compatible_surface",
            "simple_dust_mantling_interpretation_contradicted",
        ],
        default="unresolved_dust_candidate",
    )
    dust["interpretation_scope"] = "surface_environment_not_active_process"
    dust["activity_status"] = "unknown"
    dust["validation_evidence"] = [
        json.dumps({
            "albedo": None if pd.isna(a) else float(a),
            "thermal_inertia": None if pd.isna(t) else float(t),
            "adequate_for_rule": bool(ok),
        })
        for a, t, ok in zip(albedo, inertia, adequate)
    ]
    dust["validation_reason"] = np.select(
        [supports, contradicts, ~adequate],
        [
            "Separate albedo and thermal inertia measurements meet the cited fine-particulate range.",
            "Separate albedo and thermal inertia measurements oppose the simple bright, low-inertia dust model.",
            "Required non-polar albedo/thermal measurements are incomplete or outside the test scope.",
        ],
        default="Measurements are valid but fall between the conservative support and contradiction rules.",
    )
    dust["rule_id"] = "dust_thermophysical_support"
    dust["positive_process_identification"] = False
    dust["earth_matching_ready"] = False
    return dust


dust_results_df = validate_dust_candidates(candidates_df)
display(dust_results_df["validation_verdict"].value_counts(dropna=False).rename_axis("verdict").reset_index(name="candidates"))


## 2. Targeted observation lookup

ODE is used because it supports geographic searches across PDS holdings. The general PDS Search API does not
currently provide this spatial query. Longitudes are converted from −180…180 to ODE's 0…360 convention and
searches crossing 0° are split safely.


In [ ]:
retry = Retry(
    total=4,
    connect=4,
    read=4,
    backoff_factor=0.8,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
)
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "NASA-Space-Apps-Mars-validation/1.0"})
SESSION.mount("https://", HTTPAdapter(max_retries=retry))


def as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def ode_request(parameters: dict[str, Any]) -> dict[str, Any]:
    response = SESSION.get(CFG.ode_base_url, params=parameters, timeout=CFG.request_timeout_seconds)
    response.raise_for_status()
    payload = response.json()
    root = payload.get("ODEResults", {})
    if str(root.get("Status", "SUCCESS")).upper() not in {"SUCCESS", ""}:
        raise RuntimeError(f"ODE query failed: {root.get('Error', root)}")
    time.sleep(CFG.query_pause_seconds)
    return root


def extract_products(root: dict[str, Any]) -> list[dict[str, Any]]:
    container = root.get("Products", {})
    if not isinstance(container, dict):
        return []
    return [item for item in as_list(container.get("Product")) if isinstance(item, dict)]


def east_longitude(longitude: float) -> float:
    return float(longitude) % 360.0


def longitude_intervals(center_longitude: float, radius: float) -> list[tuple[float, float]]:
    center = east_longitude(center_longitude)
    low, high = center - radius, center + radius
    if low < 0:
        return [(0.0, high), (360.0 + low, 360.0)]
    if high > 360:
        return [(low, 360.0), (0.0, high - 360.0)]
    return [(low, high)]


def spatial_product_search(
    latitude: float,
    longitude: float,
    radius: float,
    instrument_host: str,
    instrument: str,
    product_type: str,
    limit: int,
) -> tuple[int, list[dict[str, Any]], list[dict[str, Any]]]:
    total_count = 0
    products_by_id: dict[str, dict[str, Any]] = {}
    query_log: list[dict[str, Any]] = []
    min_latitude = max(-90.0, float(latitude) - radius)
    max_latitude = min(90.0, float(latitude) + radius)
    for west, east in longitude_intervals(float(longitude), radius):
        params = {
            "query": "product", "target": "mars", "results": "cpm", "output": "json",
            "ihid": instrument_host, "iid": instrument, "pt": product_type,
            "minlat": min_latitude, "maxlat": max_latitude,
            "westernlon": west, "easternlon": east, "loc": "b", "limit": limit,
        }
        root = ode_request(params)
        count = int(root.get("Count", 0) or 0)
        total_count += count
        products = extract_products(root)
        for product in products:
            product_id = str(
                product.get("pdsid") or product.get("PDSID") or product.get("PDS_ID")
                or product.get("Product_id") or ""
            )
            if product_id:
                products_by_id[product_id] = product
        query_log.append({"parameters": params, "count": count, "returned": len(products)})
    return total_count, list(products_by_id.values()), query_log


def product_id(product: dict[str, Any]) -> str:
    identifier = str(
        product.get("pdsid") or product.get("PDSID") or product.get("PDS_ID")
        or product.get("Product_id") or ""
    )
    if not identifier and product.get("Product_name"):
        identifier = Path(str(product["Product_name"])).stem
    return identifier


def product_files(pds_id: str) -> list[dict[str, Any]]:
    root = ode_request({
        "query": "product", "target": "mars", "results": "f", "output": "json", "pdsid": pds_id,
    })
    files: list[dict[str, Any]] = []
    for product in extract_products(root):
        files.extend(as_list(product.get("Product_files", {}).get("Product_file")))
    return [item for item in files if isinstance(item, dict)]


## 3. CRISM hydration follow-up

For each sulfate candidate, we find the nearest returned CRISM MTRDR observation and download only its small
HYD browse image and label—not the multi-gigabyte spectral cube. The three browse bands help triage scenes,
but the official label explicitly requires full spectrum analysis for a positive mineral identification.
Therefore these candidates remain **inconclusive** in this notebook.


In [ ]:
def numeric_product_field(product: dict[str, Any], names: tuple[str, ...]) -> float:
    for name in names:
        value = product.get(name)
        if value not in (None, ""):
            try:
                return float(value)
            except (TypeError, ValueError):
                pass
    return float("nan")


def angular_distance_squared(latitude: float, longitude: float, product: dict[str, Any]) -> float:
    product_lat = numeric_product_field(product, ("Center_latitude", "center_latitude"))
    product_lon = numeric_product_field(product, ("Center_longitude", "center_longitude"))
    if not np.isfinite(product_lat) or not np.isfinite(product_lon):
        return float("inf")
    longitude_delta = abs(east_longitude(product_lon) - east_longitude(longitude))
    longitude_delta = min(longitude_delta, 360.0 - longitude_delta)
    return (product_lat - latitude) ** 2 + (longitude_delta * math.cos(math.radians(latitude))) ** 2


sulfate_candidates_df = candidates_df.loc[
    candidates_df["candidate_label"].eq("TES_sulfate_followup_candidate")
].copy()
crism_inventory_rows: list[dict[str, Any]] = []
selected_crism: list[tuple[str, str]] = []

for row in sulfate_candidates_df.itertuples(index=False):
    try:
        count, products, query_log = spatial_product_search(
            row.centroid_latitude, row.centroid_longitude, CFG.crism_search_radius_degrees,
            "MRO", "CRISM", "MTRDR", 30,
        )
        ordered = sorted(
            products,
            key=lambda product: angular_distance_squared(
                row.centroid_latitude, row.centroid_longitude, product
            ),
        )
        selected = ordered[: CFG.maximum_crism_products_per_candidate]
        selected_ids = [product_id(item) for item in selected if product_id(item)]
        selected_crism.extend((row.candidate_row_id, item) for item in selected_ids)
        crism_inventory_rows.append({
            "candidate_row_id": row.candidate_row_id,
            "region_id": row.region_id,
            "instrument": "CRISM",
            "product_type": "MTRDR",
            "search_radius_degrees": CFG.crism_search_radius_degrees,
            "coverage_count": count,
            "returned_product_ids": json.dumps([product_id(item) for item in ordered if product_id(item)]),
            "selected_product_ids": json.dumps(selected_ids),
            "coverage_count_is_lower_bound": bool(any(item["count"] >= 30 for item in query_log)),
            "query_status": "success",
            "query_detail": json.dumps(query_log),
        })
    except Exception as error:
        crism_inventory_rows.append({
            "candidate_row_id": row.candidate_row_id,
            "region_id": row.region_id,
            "instrument": "CRISM",
            "product_type": "MTRDR",
            "search_radius_degrees": CFG.crism_search_radius_degrees,
            "coverage_count": np.nan,
            "returned_product_ids": "[]",
            "selected_product_ids": "[]",
            "coverage_count_is_lower_bound": False,
            "query_status": "error",
            "query_detail": repr(error),
        })

crism_inventory_df = pd.DataFrame(crism_inventory_rows)
display(crism_inventory_df[["region_id", "coverage_count", "selected_product_ids", "query_status"]])


In [ ]:
def parse_pds_tuple(text: str, field: str) -> list[float]:
    match = re.search(rf"^\s*{re.escape(field)}\s*=\s*\((.*?)\)", text, re.MULTILINE | re.DOTALL)
    if not match:
        raise ValueError(f"Could not find {field} in PDS label")
    return [float(item.strip()) for item in match.group(1).replace("\n", " ").split(",")]


def parse_pds_scalar(text: str, field: str) -> int:
    matches = re.findall(rf"^\s*{re.escape(field)}\s*=\s*(\d+)", text, re.MULTILINE)
    if not matches:
        raise ValueError(f"Could not find {field} in PDS label")
    return int(matches[-1])


def download_small_file(url: str, destination: Path, maximum_bytes: int = 12_000_000) -> Path:
    if destination.exists() and 0 < destination.stat().st_size <= maximum_bytes:
        return destination
    with SESSION.get(url, stream=True, timeout=CFG.request_timeout_seconds) as response:
        response.raise_for_status()
        declared = int(response.headers.get("content-length", 0) or 0)
        if declared > maximum_bytes:
            raise ValueError(f"Refusing oversized follow-up file ({declared:,} bytes): {url}")
        written = 0
        with destination.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                written += len(chunk)
                if written > maximum_bytes:
                    raise ValueError(f"Download exceeded safety limit: {url}")
                handle.write(chunk)
    return destination


def hydration_file_set(pds_id: str) -> dict[str, dict[str, Any]]:
    matches: dict[str, dict[str, Any]] = {}
    for item in product_files(pds_id):
        name = str(item.get("FileName", ""))
        upper = name.upper()
        if "_BRHYDJ_MTR3." not in upper:
            continue
        suffix = Path(name).suffix.upper().lstrip(".")
        if suffix in {"IMG", "LBL"}:
            matches[suffix] = item
    return matches


def summarize_hydration_browse(pds_id: str, files: dict[str, dict[str, Any]]) -> dict[str, Any]:
    if not {"IMG", "LBL"} <= set(files):
        raise FileNotFoundError(f"HYD IMG/LBL pair unavailable for {pds_id}")
    product_directory = DATA_DIR / re.sub(r"[^A-Za-z0-9_.-]+", "_", pds_id)
    product_directory.mkdir(parents=True, exist_ok=True)
    label_path = download_small_file(str(files["LBL"]["URL"]), product_directory / str(files["LBL"]["FileName"]))
    image_path = download_small_file(str(files["IMG"]["URL"]), product_directory / str(files["IMG"]["FileName"]))
    label = label_path.read_text(encoding="latin-1")
    lines = parse_pds_scalar(label, "LINES")
    samples = parse_pds_scalar(label, "LINE_SAMPLES")
    bands = parse_pds_scalar(label, "BANDS")
    offsets = parse_pds_tuple(label, "OFFSET")
    scales = parse_pds_tuple(label, "SCALING_FACTOR")
    if bands != 3 or len(offsets) != 3 or len(scales) != 3:
        raise ValueError(f"Unexpected HYD browse layout for {pds_id}")
    raw = np.fromfile(image_path, dtype=np.uint8)
    expected = lines * samples * bands
    if raw.size != expected:
        raise ValueError(f"HYD byte count mismatch for {pds_id}: {raw.size} != {expected}")
    cube = raw.reshape(bands, lines, samples)
    physical = []
    valid_masks = []
    for band_index in range(bands):
        valid = cube[band_index] != 255
        values = offsets[band_index] + scales[band_index] * cube[band_index].astype(np.float32)
        physical.append(values)
        valid_masks.append(valid)
    joint_valid = np.logical_and.reduce(valid_masks)
    names = ["SINDEX2", "BD2100_2", "BD1900_2"]
    result: dict[str, Any] = {
        "pds_id": pds_id,
        "lines": lines,
        "samples": samples,
        "valid_pixel_fraction": float(joint_valid.mean()),
        "image_bytes": int(image_path.stat().st_size),
        "image_sha256": hashlib.sha256(image_path.read_bytes()).hexdigest(),
        "positive_mineral_identification": False,
    }
    thresholds = []
    for name, values, valid in zip(names, physical, valid_masks):
        usable = values[valid]
        if usable.size == 0:
            raise ValueError(f"No valid {name} values in {pds_id}")
        result[f"{name}__median"] = float(np.median(usable))
        result[f"{name}__q90"] = float(np.quantile(usable, 0.90))
        thresholds.append(float(np.quantile(usable, 0.90)))
    joint_upper = joint_valid.copy()
    for values, threshold in zip(physical, thresholds):
        joint_upper &= values >= threshold
    result["joint_upper_decile_fraction"] = float(joint_upper.sum() / max(joint_valid.sum(), 1))
    return result


hydration_rows: list[dict[str, Any]] = []
hydration_cache: dict[str, dict[str, Any]] = {}
for candidate_row_id, pds_id in selected_crism:
    try:
        if pds_id not in hydration_cache:
            hydration_cache[pds_id] = summarize_hydration_browse(pds_id, hydration_file_set(pds_id))
        hydration_rows.append({"candidate_row_id": candidate_row_id, "query_status": "success", **hydration_cache[pds_id]})
    except Exception as error:
        hydration_rows.append({
            "candidate_row_id": candidate_row_id,
            "pds_id": pds_id,
            "query_status": "error",
            "error": repr(error),
            "positive_mineral_identification": False,
        })

crism_hydration_metrics_df = pd.DataFrame(hydration_rows)
if len(crism_hydration_metrics_df):
    display(crism_hydration_metrics_df)
else:
    print("No CRISM HYD browse products were available in the candidate search windows.")


## 4. SHARAD follow-up inventory

The hydrogen signal is coarse. We therefore search a wider area for both radargrams and their clutter simulations.
Their presence means a radar specialist can inspect the candidate; it is not evidence that an ice interface was detected.


In [ ]:
hydrogen_candidates_df = candidates_df.loc[
    candidates_df["candidate_label"].eq("hydrogen_enrichment_candidate")
].copy()
radar_inventory_rows: list[dict[str, Any]] = []

for row in hydrogen_candidates_df.itertuples(index=False):
    for product_type in ("USRDRV2", "SHSIMU"):
        try:
            count, products, query_log = spatial_product_search(
                row.centroid_latitude, row.centroid_longitude, CFG.radar_search_radius_degrees,
                "MRO", "SHARAD", product_type, CFG.maximum_radar_ids_per_query,
            )
            ids = [product_id(item) for item in products if product_id(item)]
            radar_inventory_rows.append({
                "candidate_row_id": row.candidate_row_id,
                "region_id": row.region_id,
                "instrument": "SHARAD",
                "product_type": product_type,
                "search_radius_degrees": CFG.radar_search_radius_degrees,
                "coverage_count": count,
                "returned_product_ids": json.dumps(ids),
                "selected_product_ids": "[]",
                "coverage_count_is_lower_bound": bool(
                    any(item["count"] >= CFG.maximum_radar_ids_per_query for item in query_log)
                ),
                "query_status": "success",
                "query_detail": json.dumps(query_log),
            })
        except Exception as error:
            radar_inventory_rows.append({
                "candidate_row_id": row.candidate_row_id,
                "region_id": row.region_id,
                "instrument": "SHARAD",
                "product_type": product_type,
                "search_radius_degrees": CFG.radar_search_radius_degrees,
                "coverage_count": np.nan,
                "returned_product_ids": "[]",
                "selected_product_ids": "[]",
                "coverage_count_is_lower_bound": False,
                "query_status": "error",
                "query_detail": repr(error),
            })

radar_inventory_df = pd.DataFrame(radar_inventory_rows)
observation_inventory_df = pd.concat([crism_inventory_df, radar_inventory_df], ignore_index=True, sort=False)
successful_remote_queries = int(observation_inventory_df["query_status"].eq("success").sum())
if successful_remote_queries == 0:
    raise RuntimeError("All ODE spatial queries failed; refusing to create a misleading validation report")
display(
    radar_inventory_df.pivot_table(
        index="region_id", columns="product_type", values="coverage_count", aggfunc="first", fill_value=0
    ).reset_index()
)


## 5. Final verdicts and the Earth gate

Sulfate and hydrogen candidates remain unresolved because the required spectral/radar interpretation has not been
performed. Dust support is useful as an environmental signature, but it still does not establish active aeolian transport.
This notebook therefore cannot open process-based Earth matching yet.


In [ ]:
def unresolved_results(frame: pd.DataFrame, interpretation: str, reason: str, rule_id: str) -> pd.DataFrame:
    result = frame.copy()
    result["validation_verdict"] = "inconclusive"
    result["validated_interpretation"] = interpretation
    result["interpretation_scope"] = "candidate_requires_specialist_followup"
    result["activity_status"] = "unknown"
    result["validation_evidence"] = "{}"
    result["validation_reason"] = reason
    result["rule_id"] = rule_id
    result["positive_process_identification"] = False
    result["earth_matching_ready"] = False
    return result


sulfate_results_df = unresolved_results(
    sulfate_candidates_df,
    "unresolved_sulfate_or_hydration_candidate",
    "Nearby CRISM coverage and browse parameters only prioritize full spectral analysis; they do not identify a mineral.",
    "crism_hydration_browse_triage",
)
hydrogen_results_df = unresolved_results(
    hydrogen_candidates_df,
    "unresolved_hydrogen_enrichment_candidate",
    "Nearby SHARAD coverage only identifies available follow-up observations; no radargram/clutter interpretation was performed.",
    "sharad_coverage_inventory",
)
validation_results_df = pd.concat(
    [dust_results_df, sulfate_results_df, hydrogen_results_df], ignore_index=True, sort=False
).sort_values("candidate_row_id").reset_index(drop=True)

supported_dust = validation_results_df.loc[
    validation_results_df["validated_interpretation"].eq("fine_particulate_dust_compatible_surface")
].copy()
spatial_bins = set(
    zip(
        np.floor(supported_dust["centroid_latitude"] / 10).astype(int),
        np.floor((supported_dust["centroid_longitude"] + 180) / 20).astype(int),
    )
) if len(supported_dust) else set()
dust_environment_distribution_ready = (
    len(supported_dust) >= CFG.minimum_supported_examples_for_distribution
    and len(spatial_bins) >= CFG.minimum_spatial_bins_for_distribution
)

distribution_rows: list[dict[str, Any]] = []
distribution_features = {
    "raw_mean__albedo_mean": "dimensionless",
    "raw_mean__thermal_inertia_mean": "J m-2 K-1 s-1/2",
    "raw_mean__quality_masked__mineral_surface_dust": "dataset-specific index",
}
for feature, unit in distribution_features.items():
    if feature not in supported_dust.columns:
        continue
    values = pd.to_numeric(supported_dust[feature], errors="coerce").dropna()
    if len(values) == 0:
        continue
    distribution_rows.append({
        "interpretation": "fine_particulate_dust_compatible_surface",
        "signature_scope": "environment_not_active_process",
        "feature": feature,
        "unit": unit,
        "n": int(len(values)),
        "q05": float(values.quantile(0.05)),
        "q25": float(values.quantile(0.25)),
        "median": float(values.median()),
        "q75": float(values.quantile(0.75)),
        "q95": float(values.quantile(0.95)),
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
    })
supported_distributions_df = pd.DataFrame(distribution_rows)

earth_matching_gate = {
    "earth_matching_ready": False,
    "ready_processes": [],
    "reason": "No active or past Mars process passed all required evidence. One environment may have a reusable distribution, but environment support is not process validation.",
    "dust_environment_distribution_ready": bool(dust_environment_distribution_ready),
    "supported_dust_environment_examples": int(len(supported_dust)),
    "supported_dust_environment_spatial_bins": int(len(spatial_bins)),
    "remaining_work": {
        "aeolian_process": "morphology plus co-registered repeat-image change for current activity",
        "water_alteration": "full CRISM spectra and geological-unit association",
        "ice_related_process": "interpreted SHARAD radargrams with clutter simulation and landform context",
    },
}

verdict_summary_df = (
    validation_results_df.groupby(["candidate_label", "validation_verdict"], observed=True)
    .size().rename("candidate_count").reset_index()
)
display(verdict_summary_df)
print(json.dumps(earth_matching_gate, indent=2))


## 6. Human-readable checks

These plots answer three practical questions: what survived validation, why a dust candidate passed or failed,
and whether follow-up observations exist near the unresolved candidates.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), constrained_layout=True)

verdict_plot = verdict_summary_df.copy()
sns.barplot(
    data=verdict_plot, y="candidate_label", x="candidate_count", hue="validation_verdict",
    hue_order=["supported", "inconclusive", "rejected"],
    palette={"supported": "#15803D", "inconclusive": "#D97706", "rejected": "#B91C1C"}, ax=axes[0],
)
axes[0].set_title("Validation outcome", loc="left", fontweight="bold")
axes[0].set_xlabel("Candidate rows")
axes[0].set_ylabel("")

dust_plot = dust_results_df.dropna(subset=["raw_mean__albedo_mean", "raw_mean__thermal_inertia_mean"]).copy()
sns.scatterplot(
    data=dust_plot, x="raw_mean__thermal_inertia_mean", y="raw_mean__albedo_mean",
    hue="validation_verdict", style="validation_verdict", s=65,
    hue_order=["supported", "inconclusive", "rejected"],
    palette={"supported": "#15803D", "inconclusive": "#D97706", "rejected": "#B91C1C"}, ax=axes[1],
)
axes[1].axvspan(CFG.dust_thermal_inertia_minimum, CFG.dust_thermal_inertia_maximum, color="#15803D", alpha=0.08)
axes[1].axhline(CFG.dust_albedo_minimum, color="#334155", linestyle="--", linewidth=1)
axes[1].axvline(CFG.contradictory_thermal_inertia_minimum, color="#B91C1C", linestyle=":", linewidth=1)
axes[1].set_title("Separate thermophysical dust test", loc="left", fontweight="bold")
axes[1].set_xlabel("Thermal inertia")
axes[1].set_ylabel("Albedo")

coverage_plot = observation_inventory_df.copy()
coverage_plot["query"] = coverage_plot["instrument"] + " " + coverage_plot["product_type"]
sns.boxplot(data=coverage_plot, x="coverage_count", y="query", color="#60A5FA", ax=axes[2])
sns.stripplot(data=coverage_plot, x="coverage_count", y="query", color="#172554", size=5, ax=axes[2])
axes[2].set_title("Available targeted observations", loc="left", fontweight="bold")
axes[2].set_xlabel("Products returned (capped; may be a lower bound)")
axes[2].set_ylabel("")

figure_path = FIGURE_DIR / "candidate_validation_overview.png"
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.show()

map_df = validation_results_df.copy()
fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
sns.scatterplot(
    data=map_df, x="centroid_longitude", y="centroid_latitude",
    hue="candidate_label", style="validation_verdict", size="screening_score",
    sizes=(35, 180), alpha=0.8, ax=ax,
)
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xticks(np.arange(-180, 181, 60)); ax.set_yticks(np.arange(-90, 91, 30))
ax.set_title("Where the reviewed Mars candidates are", loc="left", fontweight="bold")
ax.set_xlabel("Longitude (degrees)"); ax.set_ylabel("Latitude (degrees)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
map_path = FIGURE_DIR / "candidate_locations.png"
fig.savefig(map_path, dpi=170, bbox_inches="tight")
plt.show()


In [ ]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if not isinstance(value, (str, bytes)) else False:
        return None
    return value


export_columns = [
    "candidate_row_id", "process_id", "candidate_label", "region_id", "neutral_type", "scale",
    "analysis_cohort", "centroid_latitude", "centroid_longitude", "screening_score",
    "validation_verdict", "validated_interpretation", "interpretation_scope", "activity_status",
    "validation_evidence", "validation_reason", "rule_id", "positive_process_identification",
    "earth_matching_ready", "missing_for_validation", "alternative_explanations",
    "raw_mean__albedo_mean", "raw_mean__thermal_inertia_mean",
    "quality_mean__albedo_coverage", "quality_mean__thermal_inertia_coverage",
    "quality_mean__polar_metric_limit", "raw_mean__quality_masked__mineral_surface_dust",
]
export_columns = [column for column in export_columns if column in validation_results_df.columns]
validation_results_df[export_columns].to_parquet(
    RUN_ROOT / "mars_candidate_validation_results.parquet", index=False, compression="zstd"
)
observation_inventory_df.to_parquet(
    RUN_ROOT / "mars_validation_observation_inventory.parquet", index=False, compression="zstd"
)
crism_hydration_metrics_df.to_parquet(
    RUN_ROOT / "mars_crism_hydration_metrics.parquet", index=False, compression="zstd"
)
supported_distributions_df.to_parquet(
    RUN_ROOT / "mars_supported_environment_distributions.parquet", index=False, compression="zstd"
)
validation_rules_df.to_csv(RUN_ROOT / "mars_validation_rules.csv", index=False)
(RUN_ROOT / "earth_matching_gate.json").write_text(
    json.dumps(json_safe(earth_matching_gate), indent=2), encoding="utf-8"
)

assert len(validation_results_df) == len(review_queue_df), "Every queued candidate must receive one verdict"
assert validation_results_df["candidate_row_id"].is_unique
assert set(validation_results_df["validation_verdict"]) <= {"supported", "rejected", "inconclusive"}
assert not validation_results_df.loc[
    validation_results_df["candidate_label"].isin(
        ["TES_sulfate_followup_candidate", "hydrogen_enrichment_candidate"]
    ), "positive_process_identification"
].any(), "Coverage/browse products must not become positive process identifications"
assert not validation_results_df["earth_matching_ready"].any()
assert earth_matching_gate["earth_matching_ready"] is False
if len(crism_hydration_metrics_df):
    assert not crism_hydration_metrics_df["positive_mineral_identification"].fillna(False).any()

summary = {
    "notebook": "07_mars_candidate_validation",
    "input_queue": str(QUEUE_PATH),
    "input_regions": str(REGIONS_PATH),
    "candidate_rows": int(len(validation_results_df)),
    "unique_regions": int(validation_results_df["region_id"].nunique()),
    "verdict_counts": validation_results_df["validation_verdict"].value_counts().to_dict(),
    "successful_ode_queries": successful_remote_queries,
    "failed_ode_queries": int(observation_inventory_df["query_status"].eq("error").sum()),
    "unique_crism_products_analyzed": int(crism_hydration_metrics_df.get("pds_id", pd.Series(dtype=str)).nunique()),
    "earth_matching_gate": earth_matching_gate,
    "config": asdict(CFG),
}
(RUN_ROOT / "run_summary.json").write_text(
    json.dumps(json_safe(summary), indent=2), encoding="utf-8"
)
summary_markdown = "\n".join([
    "# Mars candidate validation summary",
    "",
    f"- Candidate rows reviewed: {summary['candidate_rows']}",
    f"- Unique regions: {summary['unique_regions']}",
    f"- Supported dust-compatible environments: {summary['verdict_counts'].get('supported', 0)}",
    f"- Rejected by the current independent test: {summary['verdict_counts'].get('rejected', 0)}",
    f"- Inconclusive: {summary['verdict_counts'].get('inconclusive', 0)}",
    f"- Successful ODE inventory queries: {summary['successful_ode_queries']}",
    "- Earth process matching ready: **No**",
    "",
    "Supported is limited to a fine-particulate dust-compatible surface environment. It is not proof of active wind transport.",
    "CRISM hydration browse products and SHARAD coverage are follow-up inventories, not positive mineral or ice identifications.",
])
(RUN_ROOT / "run_summary.md").write_text(summary_markdown + "\n", encoding="utf-8")

print("Validation exports written:")
for path in sorted(RUN_ROOT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RUN_ROOT)} ({path.stat().st_size:,} bytes)")
print("\nTL;DR:", json.dumps(json_safe(summary["verdict_counts"])), "— Earth process matching remains closed.")
